In [1]:
import pandas as pd
import torch
from bert_score import score
from pathlib import Path


c:\Users\belen\reviews-aa\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
carpeta = Path("reseñas") if Path("reseñas").exists()else Path(".")

In [4]:
df_es = pd.read_csv(carpeta / "resenas_espanol_trad.csv")
df_pt = pd.read_csv(carpeta / "resenas_portugues_trad.csv")

In [5]:
print("Dataset español:", df_es.shape)
print("Dataset portugués:", df_pt.shape)

Dataset español: (6632, 4)
Dataset portugués: (6632, 4)


In [6]:
muestra_es = df_es.sample(frac=0.10, random_state=42).copy()
muestra_pt = df_pt.sample(frac=0.10, random_state=42).copy()

In [7]:
print("Muestra español:", len(muestra_es))
print("Muestra portugués:", len(muestra_pt))

Muestra español: 663
Muestra portugués: 663


In [9]:
def calcular_bertscore(df, columna_traduccion):

    originales = df["Review Text"].astype(str).tolist()
    traducciones = df[columna_traduccion].astype(str).tolist()

    P, R, F1 = score(
        traducciones,
        originales,
        model_type="bert-base-multilingual-cased",
        num_layers=9,
        batch_size=8,
        device="cuda" if torch.cuda.is_available() else "cpu",
        verbose=True
    )

    resultados = df.copy()

    resultados["BERTScore_P"] = P.cpu().numpy()
    resultados["BERTScore_R"] = R.cpu().numpy()
    resultados["BERTScore_F1"] = F1.cpu().numpy()

    return resultados


In [10]:
resultado_es = calcular_bertscore(
    muestra_es, "Review Text_es"
)



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 441.58it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


100%|██████████| 166/166 [02:12<00:00,  1.25it/s]


computing greedy matching.


100%|██████████| 83/83 [00:01<00:00, 65.14it/s]


done in 134.02 seconds, 4.95 sentences/sec


In [12]:
resultado_pt = calcular_bertscore(
    muestra_pt, "Review Text_pt-BR"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 426.86it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


100%|██████████| 166/166 [02:11<00:00,  1.27it/s]


computing greedy matching.


100%|██████████| 83/83 [00:01<00:00, 68.94it/s]


done in 132.38 seconds, 5.01 sentences/sec


In [13]:
def mostrar_metricas(df):
    resumen = pd.DataFrame({
        "Precision": [df["BERTScore_P"].mean()],
        "Recall": [df["BERTScore_R"].mean()],
        "F1": [df["BERTScore_F1"].mean()]
    })

    display(resumen.round(4))

In [14]:
mostrar_metricas(resultado_es)

,Precision,Recall,F1
0,0.749,0.758,0.7534


In [15]:
mostrar_metricas(resultado_pt)

,Precision,Recall,F1
0,0.7484,0.7605,0.7543


In [24]:
muestra_es_1 = df_es.sample(frac=0.01, random_state=42).copy()
muestra_pt_1 = df_pt.sample(frac=0.01, random_state=42).copy()

In [25]:
pd.set_option("display.max_colwidth", None)
muestra_es_1

,Rating,Date,Review Text,Review Text_es
4554,5,4/20/2009,"The #1 best thing about Jarlings is the speedy, friendly service. Well, at least it was the one whole time I've been there. There were a ton of people there, but the line moved very quickly (which was good for everyone nearby--I had serious custard jones). #2 best thing is, of course, the custard. I had a 'mini' butterscotch sundae, which was *plenty* if you just want a little something. Mini my foot, really. It's a large scoop of custard. SO worth it for the meager price tag of $2.55. #3 best thing is all the fixings available.....like butter pecans or pecan pralines. Or both. I'm drooling just thinking about that. And I cannot *wait* to try a ""Snickerette Snowstorm."" Mmm....bad for me...","Lo mejor de Jarlings, sin duda, es su servicio rápido y amable. Bueno, al menos así fue durante todo el tiempo que estuve allí. Había mucha gente, pero la cola se movía muy rápido (lo cual era bueno para todos los que estaban cerca, ya que yo tenía una fuerte necesidad de helado). En segundo lugar, por supuesto, está el helado. Tuve un ""mini"" sundae de mantequilla de nuez, que era *más que suficiente* si solo quieres algo pequeño. ""Mini"", ¿en serio? Es una porción grande de helado. ¡Totalmente vale la pena por el bajo precio de 2,55 dólares! En tercer lugar, lo mejor es la variedad de ingredientes disponibles... como nueces con mantequilla o pralinés de nuez. O ambos. ¡Estoy imaginándome ya! Y no puedo esperar a probar un ""Snickerette Snowstorm"". ¡Mmm... no es lo más saludable!"
4864,4,3/27/2015,"From reading other reviews (and my sister's personal experience), I was worried the line would be extremely long but we waited only about 15 minutes on a Friday night around 8:45pm. We ordered LOTS of desserts - chocolate eclair ($3.49), cream puff ($2.09), mixed berry 4"" tart ($4.89), lemon raspberry fruit tart ($4.89), mini chocolate ganache cake (?), salted caramel brownies ($2.29), coconut macaroons ($1.59), pistachio flavored petit-four ($2.09), espresso mini mousse cups ($1.59), and every flavor of french macarons they had available ($2.29).. They had many workers restocking, but only 1 person taking orders and 1 person working the cash register. Is this to regulate the flow of the line? It must slow down the line because the person taking the order has to tell the cashier everything in the order, and if the cashier doesn't hear everything, then it's just chaos. Any way, I personally didn't try everything we ordered so I can only give my sweet-tooth loving opinion on some of the items..but the chocolate ganache cake is highly recommendable. Not too rich and I loved the filling. It can be split between two people. The espresso mini mousse cup was not as great as it sounds. Bite-size and not as fluffy as I wanted. The salted caramel brownie is, if not the best (read in many other reviews), one of the most popular items.. but I thought the caramel was way too much. It is what contains the salt of course and the brownie would just be a normal chocolate brownie without it.. because the brownie alone did not taste like any chocolate worth remembering. Lastly, the cream puff is filled with custard which is something I love, but it isn't worth $2. I mean, it's huge, so if you really love cream puffs, have at it. Order it alone or split it with someone and order one other item.","Después de leer otras reseñas (y la experiencia personal de mi hermana), me preocupaba que la cola fuera muy larga, pero en una noche de viernes alrededor de las 20:45, tardamos solo unos 15 minutos. Pedimos MUCHOS postres: éclair de chocolate ($3.49), tartaleta de crema ($2.09), tartaleta de frutos rojos de 4 pulgadas ($4.89), tartaleta de limón y frambuesa ($4.89), pastelito de ganache de chocolate pequeño (?), brownies de caramelo salado ($2.29), galletas de coco ($1.59), petit-four de pistacho ($2.09), mini vasos de mousse de espresso ($1.59) y todos los sabores de macarons franceses que tenían disponible

In [ ]:
pd.set_option("display.max_colwidth", None)
muestra_pt_1

,Rating,Date,Review Text,Review Text_pt-BR
4554,5,6/12/2018,Best Stromboli and cannoli. Excellent customer service and don't miss the gelato. Parking may be little struggle but you will definitely find a spot. Their cookies is delicious,"Melhor Stromboli e cannoli. Ótimo atendimento ao cliente e não deixe de experimentar o sorvete. A procura por vagas de estacionamento pode ser um pouco difícil, mas certamente encontrará um lugar. Seus biscoitos são deliciosos."
4864,5,3/17/2022,"I loved this ice cream, just the smell of this place is fantastic, reasonable prices too!!!","Eu adorei este sorvete, o aroma deste lugar é fantástico, e os preços também são razoáveis!!!"
5099,1,10/7/2018,"Staff was very rude! Me and my husband waited by the register forever to order and no one acknowledged us. we stood there for a couple minutes with 4 employees standing behind the counter only one was helping a guest. They have a lot of toys in the store, my husband was looking at one that was already out of the box and she accused him of opening it and wanted us to pay for it. That was all the interaction we had with the staff. We moved back over by the Register and waited for a few minutes longer but still nothing and they walked by us twice without even a ""I will be right with you"". At that point we left.","A equipe foi muito mal-educada! Eu e meu marido ficamos esperando na fila por muito tempo para fazer a compra, e ninguém prestou atenção a nós. Ficamos ali por alguns minutos, com 4 funcionários atrás do balcão, mas apenas um estava atendendo um cliente. A loja tem muitos brinquedos, e meu marido estava olhando um que já estava aberto, e uma funcionária o acusou de ter aberto e exigiu que pagássemos por ele. Essa foi a única interação que tivemos com a equipe. Voltamos para perto do balcão e esperamos por mais alguns minutos, mas nada, e eles passaram por nós duas vezes sem sequer dizer ""estou indo"". Nesse ponto, decidimos ir embora."
4730,5,4/21/2019,I've always wanted to try this place and believe me the wait was worth it. I went on my birthday and I loved it! Probably once on the best places to get unique milkshakes and maroons. You would think some of the combos are weird but they taste amazing together. The maroon are big and sweet enough to share. If you want to talk about sugar overload this is the place. They have food but I just went for the ice cream.,"Eu sempre quis experimentar este lugar e, acredite, a espera valeu a pena. Fui no meu aniversário e adorei! Provavelmente um dos melhores lugares para experimentar milkshakes e sorvetes únicos. Você pode achar que alguns dos combos são estranhos, mas eles combinam de forma incrível. Os sorvetes são grandes e doces o suficiente para compartilhar. Se você quer falar sobre excesso de açúcar, este é o lugar. Eles também têm comida, mas eu fui apenas pelo sorvete."
5998,5,3/25/2018,This place is great for late late night sweet treats. We love going here. The decor is so eclectic it makes you feel like you are somewhere else. The treats are great as well as the coffee. Will definitely be going back.,Este lugar é perfeito para aproveitar doces e sobremesas até tarde da noite. Adoramos ir aqui. A decoração é tão eclética que faz você se sentir em outro lugar. Os doces e o café também são excelentes. Com certeza voltaremos.
...,...,...,...,...
4602,4,2/1/2013,"This bakery is just a little to-go place, in the middle of the Venetian hotel. They are separate from the Bouchon restaurant (which is upstairs). Their selection is limited unlike their other bakery in Napa. However I make good use of their daily selection of macarons, croissants and pastries. The macarons are huge but their selection is limited with just the basic flavors. When you do have a chance to go to Napa, pls stop by their bakery there. I love their seasonal flavors and my favorite was the Cherry Cheesecake macaron and Monkey Bread. The male barista during my visit was a bit overwhelmed. I ordered chai tea to go and he made 

: 